In [30]:
import pandas as pd
import csv

In [9]:
def into_df(input):
    return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True)

In [12]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_COMPONENT.csv"
df = into_df(input_path)
df.columns


Index(['COMPONENT_ID', 'COMPONENT_NUMBER', 'PRODUCT_ID', 'LICENCE_ID',
       'COMPONENT_DESCRIPTION', 'WEIGHT_OF_DIVIDED_PREPARATION',
       'MAX_DAILY_DOSE', 'MAX_DAILY_DOSE_UNIT', 'MAX_SINGLE_DOSE',
       'MAX_SINGLE_DOSE_UNIT', 'VISUAL_IDENTIFICATION', 'DOSAGE_FORM_CODE',
       'ADDITIONAL_INFO', 'CREATION_DATE', 'LAST_UPDATE_DATE'],
      dtype='object')

In [13]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_FORMULATION.csv"
df = into_df(input_path)
df.columns

Index(['FORMULATION_ID', 'COMPONENT_ID', 'FORMULATION_TYPE', 'INGREDIENT_ID',
       'LABEL_NAME_AND_POTENCY', 'CONCENTRATION_RATIO_1',
       'CONCENTRATION_RATIO_2', 'POTENCY', 'POTENCY_MEASURED',
       'UNIT_PROPORTION_CODE', 'ADDITIONAL_INFORMATION', 'STANDARDISED_FLAG',
       'CREATION_DATE', 'LAST_UPDATE_DATE', 'QUANTITY',
       'FORMULATION_INGREDIENT_TYPE'],
      dtype='object')

In [14]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_INGREDIENT.csv"
df = into_df(input_path)
df.columns

Index(['INGREDIENT_ID', 'INGREDIENT_NAME', 'INGREDIENT_CATEGORY_CODE',
       'CAS_NUMBER'],
      dtype='object')

In [18]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_LICENCE.csv"
df = into_df(input_path)
df.columns
# df.head()

/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_60368/3108688076.py:2: DtypeWarning: Columns (19,20,24,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True)


Index(['LICENCE_ID', 'LICENCE_NAME', 'LICENCE_IDENTIFIER', 'LICENCE_STATUS',
       'LICENCE_STATUS_EFFECTIVE_DATE', 'LICENCE_STATUS_CHANGED_BY',
       'LICENCE_STATUS_REASON', 'SPONSOR_ID', 'SPONSOR_LOCATION_FUNCTION_ID',
       'SPONSOR_DATE_EFFECTIVE', 'AGENT_ID', 'AGENT_LOCATION_FUNCTION_ID',
       'AGENT_DATE_EFFECTIVE', 'LICENCE_START_DATE', 'LICENCE_CANCELLED_DATE',
       'LICENCE_ISSUED_DATE', 'LICENCE_TYPE', 'LICENCE_THERAPEUTIC_TYPE',
       'LICENCE_CLASS', 'LICENCE_ORIGIN', 'LICENCE_ORIGIN_START_DATE',
       'LICENCE_ORIGIN_ID', 'LICENCE_PROV_END_DATE',
       'LICENCE_PRODUCT_CATEGORY', 'TYPE_OF_THERAPEUTIC_GOOD', 'CHARGE_LEVEL',
       'CHARGE_EFFECTIVE_DATE', 'BMU_EXEMPTION_STATUS', 'BMU_RENEWAL_DATE',
       'BMU_BILLING_LOC_FUNCTION_ID', 'REGISTRATION_TYPE', 'CODE_STOCK_FLAG',
       'CODE_STOCK_LICENCE_ID', 'MANUF_SPONSOR_CONFIDENTIALITY',
       'MANUF_AGENT_CONFIDENTIALITY', 'FORMUL_SPONSOR_CONFIDENTIALITY',
       'FORMUL_AGENT_CONFIDENTIALITY', 'ADG_CODE', 'AP

In [23]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_PRODUCT.csv"
df = into_df(input_path)
print(len(df))
df.columns

31942


Index(['PRODUCT_ID', 'PRODUCT_NAME', 'LICENCE_ID', 'PRODUCT_STATUS',
       'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE',
       'PRODUCT_CEASED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO',
       'ADDITIONAL_WARNING_INFO', 'PRODUCT_CODE', 'CREATION_DATE',
       'LAST_UPDATE_DATE'],
      dtype='object')

In [21]:
input_path = "./../../data/AusPAR/COGNOS_V_GEN_SPECIFIC_INDIC.csv"
df = into_df(input_path)
print(len(df))
df.columns


182289


Index(['PRODUCT_ID', 'INDICATION_TEXT', 'CREATION_DATE', 'LAST_UPDATE_DATE'], dtype='object')

In [32]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# CONFIG – adjust once
# ------------------------------------------------------------------
DATA_DIR   = Path("./../../data/AusPAR")   # folder with cleaned CSVs
FILE_GLOB  = "*.csv"                                 # pattern to pick which files
SAVE_MERGED = True
MERGED_OUT = DATA_DIR / "AusPAR_merged.csv"

# ------------------------------------------------------------------
# 1. discover files
# ------------------------------------------------------------------
csv_paths = sorted(DATA_DIR.glob(FILE_GLOB))
if not csv_paths:
    raise FileNotFoundError(f"No files matching {FILE_GLOB} in {DATA_DIR}")

# ------------------------------------------------------------------
# 2. load each file, keeping only those with PRODUCT_ID
# ------------------------------------------------------------------
dfs            = []
schema_summary = {}            # file → set(columns)
skipped_files  = []            # files without PRODUCT_ID

for p in csv_paths:
    df = into_df(p)                         # your helper
    if "PRODUCT_ID" not in df.columns:
        skipped_files.append(p.name)
        continue                            # ignore this file

    dfs.append(df)
    schema_summary[p.name] = set(df.columns)

if not dfs:
    raise RuntimeError("No CSV contained a 'PRODUCT_ID' column — nothing to merge.")

print(f"✓ will merge {len(dfs)} file(s) that contain PRODUCT_ID")
if skipped_files:
    print("✗ skipped (no PRODUCT_ID):", ", ".join(skipped_files))

# ------------------------------------------------------------------
# 3. analyse schema differences
# ------------------------------------------------------------------
all_cols   = set().union(*schema_summary.values())
common_cols = set.intersection(*schema_summary.values())
print("┌ Schema check")
print(f"│  files found     : {len(csv_paths)}")
print(f"│  union of columns: {len(all_cols)}")
print(f"│  common to all   : {len(common_cols)}")
print("└──────────────────")

for fname, cols in schema_summary.items():
    missing = all_cols - cols
    if missing:
        print(f" ⚠ {fname:30s}  missing {len(missing):2d} → {sorted(missing)}")

# ------------------------------------------------------------------
# 4. normalise & merge
# ------------------------------------------------------------------
normed = []
for df in dfs:
    # add any absent columns as NaN so every df has identical schema
    for col in all_cols - set(df.columns):
        df[col] = pd.NA
    normed.append(df[sorted(all_cols)])   # identical column order

merged_df = pd.concat(normed, ignore_index=True)

print(f"\n✅ merged shape: {merged_df.shape}")

if SAVE_MERGED:
    # merged_df.to_csv(MERGED_OUT, index=False, sep="~")   # keep tilde delimiter
    merged_df.to_csv(
        MERGED_OUT,
        index=False,
        sep=",",                   # standard CSV
        # encoding="utf-8-sig",      # adds BOM for Excel
        quoting=csv.QUOTE_MINIMAL  # only quote when needed
    )
    print(f"📄 written to: {MERGED_OUT.resolve()}")


/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_60368/3108688076.py:2: DtypeWarning: Columns (19,20,24,28,37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(input, delimiter='~', encoding='utf-8', skipinitialspace=True)


✓ will merge 3 file(s) that contain PRODUCT_ID
✗ skipped (no PRODUCT_ID): COGNOS_V_GEN_FORMULATION.csv, COGNOS_V_GEN_INGREDIENT.csv, COGNOS_V_GEN_LICENCE.csv
┌ Schema check
│  files found     : 6
│  union of columns: 25
│  common to all   : 3
└──────────────────
 ⚠ COGNOS_V_GEN_COMPONENT.csv      missing 10 → ['ADDITIONAL_WARNING_INFO', 'INDICATION_TEXT', 'PRODUCT_CEASED_DATE', 'PRODUCT_CODE', 'PRODUCT_NAME', 'PRODUCT_STATUS', 'PRODUCT_STATUS_EFFECTIVE_DATE', 'PRODUCT_SUPPLIED_DATE', 'PRODUCT_TYPE', 'SHELF_LIFE_CONTAINER_INFO']
 ⚠ COGNOS_V_GEN_PRODUCT.csv        missing 12 → ['ADDITIONAL_INFO', 'COMPONENT_DESCRIPTION', 'COMPONENT_ID', 'COMPONENT_NUMBER', 'DOSAGE_FORM_CODE', 'INDICATION_TEXT', 'MAX_DAILY_DOSE', 'MAX_DAILY_DOSE_UNIT', 'MAX_SINGLE_DOSE', 'MAX_SINGLE_DOSE_UNIT', 'VISUAL_IDENTIFICATION', 'WEIGHT_OF_DIVIDED_PREPARATION']
 ⚠ COGNOS_V_GEN_SPECIFIC_INDIC.csv  missing 21 → ['ADDITIONAL_INFO', 'ADDITIONAL_WARNING_INFO', 'COMPONENT_DESCRIPTION', 'COMPONENT_ID', 'COMPONENT_NUMBER',

/var/folders/c_/d2jd7yn50y93c2sqyws6yswc0000gn/T/ipykernel_60368/2458197599.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged_df = pd.concat(normed, ignore_index=True)



✅ merged shape: (247377, 25)
📄 written to: /Users/shtosti/Dropbox/Projects/DrugFork/data/AusPAR/AusPAR_merged.csv
